In [1]:
!pip install -q datasets huggingface_hub pandas
print("Installed.")

Installed.


In [2]:
from huggingface_hub import hf_hub_download
import pandas as pd

print("Downloading Pexels URL catalog from HuggingFace...")
parquet_path = hf_hub_download(
    repo_id="Corran/pexelvideos",
    filename="PexelVideos.parquet.gzip",
    repo_type="dataset"
)
print(f"Downloaded to: {parquet_path}")

df = pd.read_parquet(parquet_path)
print(f"\nTotal Pexels URLs in catalog: {len(df)}")
print(f"\nColumns:")
for c in df.columns:
    print(f"  {c}: {df[c].dtype}")

print(f"\nFirst row (transposed for readability):")
print(df.iloc[0].to_string())

PexelVideos.parquet.gzip:   0%|          | 0.00/29.9M [00:00<?, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--Corran--pexelvideos/snapshots/a9f7f1ac75934a7c01d3ca02217544251939c881/PexelVideos.parquet.gzip

Total Pexels URLs in catalog: 358551

Columns:
  loc: object
  thumbnail_loc: object
  title: object
  description: object
  content_loc: object
  player_loc: object
  duration: object
  view_count: object
  publication_date: object
  family_friendly: object
  requires_subscription: object
  uploader_info: object
  live: object

First row (transposed for readability):
loc                      https://www.pexels.com/video/mother-and-two-ki...
thumbnail_loc            https://images.pexels.com/videos/7603862/adult...
title                    Mother and Two Kids Sitting on Red Couch · Fre...
description              One of many great free stock videos from Pexel...
content_loc              https://player.vimeo.com/external/539902263.hd...
player_loc                                     https://vimeo.com/539902263
duration                    

In [6]:
import requests
import random
import time


API_KEY = "abc"

HEADERS = {"Authorization": API_KEY}
SEARCH_URL = "https://api.pexels.com/videos/search"

# Category → search queries
category_queries = {
    "portrait": ["woman smiling face", "man talking close up", "person portrait"],
    "hands": ["hands typing", "chopping vegetables", "hands cooking"],
    "multi_person": ["friends walking", "family dinner", "couple conversation"],
    "motion": ["running person", "kicking soccer ball", "skateboarding"],
    "text_scene": ["bookstore sign", "cafe menu", "chalkboard classroom"],
    "texture": ["chef cooking", "pottery wheel", "rain window"],
    "animal": ["dog running", "cat close up", "horse galloping"],
    "edge_case": ["dancer stage", "juggler", "carnival"],
}

TARGET_PER_CATEGORY = 8
random.seed(42)

selected_pexels = []
all_seen_ids = set()

for our_category, queries in category_queries.items():
    picked = []
    for query in queries:
        if len(picked) >= TARGET_PER_CATEGORY:
            break
        
        params = {
            "query": query,
            "orientation": "landscape",
            "per_page": 30,
            "page": 1,
        }
        
        try:
            r = requests.get(SEARCH_URL, headers=HEADERS, params=params, timeout=15)
            r.raise_for_status()
        except Exception as e:
            print(f"  {our_category} / '{query}': API error {e}")
            continue
        
        data = r.json()
        videos = data.get("videos", [])
        
        candidates = []
        for v in videos:
            if v["id"] in all_seen_ids:
                continue
            if v["duration"] < 3 or v["duration"] > 30:
                continue
            hd_files = [f for f in v["video_files"] 
                       if f.get("width", 0) >= 1280 and f.get("file_type") == "video/mp4"]
            if not hd_files:
                continue
            best = sorted(hd_files, key=lambda f: f["width"])[0]
            candidates.append({
                "pexels_id": v["id"],
                "our_category": our_category,
                "matched_query": query,
                "pexels_page_url": v["url"],
                "video_url": best["link"],
                "video_width": best["width"],
                "video_height": best["height"],
                "duration": v["duration"],
                "user_name": v["user"]["name"],
                "user_url": v["user"]["url"],
            })
        
        n_take = min(3, TARGET_PER_CATEGORY - len(picked), len(candidates))
        if n_take > 0:
            picks = random.sample(candidates, n_take)
            for p in picks:
                picked.append(p)
                all_seen_ids.add(p["pexels_id"])
        
        time.sleep(0.3)
    
    selected_pexels.extend(picked)
    print(f"{our_category}: {len(picked)}/{TARGET_PER_CATEGORY} selected")

print(f"\nTotal selected: {len(selected_pexels)} videos")

portrait: 8/8 selected
hands: 8/8 selected
multi_person: 8/8 selected
motion: 8/8 selected
text_scene: 8/8 selected
texture: 8/8 selected
animal: 8/8 selected
edge_case: 8/8 selected

Total selected: 64 videos


In [7]:
from pathlib import Path
import urllib.request
import time

DOWNLOAD_ROOT = Path("/kaggle/working/pexels_downloads")
DOWNLOAD_ROOT.mkdir(exist_ok=True)

# Clear any leftover partial downloads
for f in DOWNLOAD_ROOT.glob("*.mp4"):
    if f.stat().st_size < 100_000:
        f.unlink()

opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'Academic Research (MSc Dissertation)')]
urllib.request.install_opener(opener)

failed = []
successful = 0

for i, item in enumerate(selected_pexels):
    filename = f"pexels_{item['our_category']}_{item['pexels_id']}.mp4"
    dst = DOWNLOAD_ROOT / filename
    
    if dst.exists() and dst.stat().st_size > 100_000:
        item['video_id'] = filename
        successful += 1
        continue
    
    try:
        urllib.request.urlretrieve(item['video_url'], str(dst))
        if dst.stat().st_size < 100_000:
            raise Exception(f"File too small ({dst.stat().st_size} bytes)")
        item['video_id'] = filename
        successful += 1
        size_mb = dst.stat().st_size / 1e6
        print(f"  [{i+1}/{len(selected_pexels)}] {filename} ({size_mb:.1f} MB)")
    except Exception as e:
        failed.append((filename, str(e)[:120]))
        print(f"  [{i+1}/{len(selected_pexels)}] FAILED: {str(e)[:80]}")
        if dst.exists():
            dst.unlink()
    
    time.sleep(1.0)

print(f"\n{'='*60}")
print(f"Downloads complete. Successful: {successful}/{len(selected_pexels)}")
print(f"Failed: {len(failed)}")
if failed:
    print("\nFailures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

  [1/64] pexels_portrait_8724519.mp4 (2.3 MB)
  [2/64] pexels_portrait_8724243.mp4 (2.6 MB)
  [3/64] pexels_portrait_5045936.mp4 (1.5 MB)
  [4/64] pexels_portrait_5051072.mp4 (2.3 MB)
  [5/64] pexels_portrait_6099467.mp4 (4.9 MB)
  [6/64] pexels_portrait_7735868.mp4 (3.8 MB)
  [7/64] pexels_portrait_4057145.mp4 (6.2 MB)
  [8/64] pexels_portrait_10970723.mp4 (3.1 MB)
  [9/64] pexels_hands_2369564.mp4 (2.8 MB)
  [10/64] pexels_hands_8888977.mp4 (5.7 MB)
  [11/64] pexels_hands_1326864.mp4 (3.9 MB)
  [12/64] pexels_hands_8625868.mp4 (5.0 MB)
  [13/64] pexels_hands_37239368.mp4 (6.8 MB)
  [14/64] pexels_hands_4201451.mp4 (2.7 MB)
  [15/64] pexels_hands_10048810.mp4 (3.2 MB)
  [16/64] pexels_hands_11102752.mp4 (2.9 MB)
  [17/64] pexels_multi_person_8774198.mp4 (4.6 MB)
  [18/64] pexels_multi_person_7325946.mp4 (3.1 MB)
  [19/64] pexels_multi_person_8121423.mp4 (6.8 MB)
  [20/64] pexels_multi_person_6305116.mp4 (2.3 MB)
  [21/64] pexels_multi_person_6305179.mp4 (2.7 MB)
  [22/64] pexels_multi

In [8]:
import subprocess
from pathlib import Path

NORMALISED_ROOT = Path("/kaggle/working/pexels_normalised")
NORMALISED_ROOT.mkdir(exist_ok=True)

downloaded_items = [item for item in selected_pexels 
                   if 'video_id' in item and (DOWNLOAD_ROOT / item['video_id']).exists()]
print(f"Normalising {len(downloaded_items)} videos to 832×480, 24 fps, 3 sec...\n")

TARGET_W, TARGET_H = 832, 480
TARGET_FPS = 24
TARGET_DURATION = 3

failed_norm = []
for i, item in enumerate(downloaded_items):
    src = DOWNLOAD_ROOT / item['video_id']
    dst = NORMALISED_ROOT / item['video_id']
    
    cmd = [
        "ffmpeg", "-y", "-i", str(src),
        "-t", str(TARGET_DURATION),
        "-vf", f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=decrease,"
               f"pad={TARGET_W}:{TARGET_H}:(ow-iw)/2:(oh-ih)/2:color=black",
        "-r", str(TARGET_FPS),
        "-c:v", "libx264", "-preset", "fast", "-crf", "23",
        "-an",
        str(dst)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        failed_norm.append((item['video_id'], result.stderr[-200:]))
    if (i+1) % 10 == 0:
        print(f"  Normalised {i+1}/{len(downloaded_items)}")

print(f"\nNormalisation complete.")
print(f"Successful: {len(downloaded_items) - len(failed_norm)}")
print(f"Failed: {len(failed_norm)}")
if failed_norm:
    print("\nFailures:")
    for name, err in failed_norm[:5]:
        print(f"  {name}: {err[:150]}")

Normalising 64 videos to 832×480, 24 fps, 3 sec...

  Normalised 10/64
  Normalised 20/64
  Normalised 30/64
  Normalised 40/64
  Normalised 50/64
  Normalised 60/64

Normalisation complete.
Successful: 64
Failed: 0


In [9]:
import json
from collections import Counter

final_pexels = []
for item in selected_pexels:
    if 'video_id' not in item:
        continue
    if (NORMALISED_ROOT / item['video_id']).exists():
        item['source'] = "Pexels API"
        item['license'] = "Pexels License (free for commercial and research use, no attribution required; creator credited in user_name for transparency)"
        item['normalised_to'] = {
            "width": TARGET_W, "height": TARGET_H,
            "fps": TARGET_FPS, "duration_sec": TARGET_DURATION,
            "codec": "libx264", "crf": 23, "audio": "stripped"
        }
        # Remove the raw video_url — it's a signed CDN link that will expire
        item.pop('video_url', None)
        final_pexels.append(item)

METADATA_PATH = Path("/kaggle/working/pexels_metadata.json")
with open(METADATA_PATH, "w") as f:
    json.dump(final_pexels, f, indent=2, default=str)

counts = Counter(item['our_category'] for item in final_pexels)
print(f"Final corpus: {len(final_pexels)} videos\n")
print("Breakdown by category:")
for cat, n in sorted(counts.items()):
    print(f"  {cat}: {n}")
print(f"\nMetadata: {METADATA_PATH}")

Final corpus: 64 videos

Breakdown by category:
  animal: 8
  edge_case: 8
  hands: 8
  motion: 8
  multi_person: 8
  portrait: 8
  text_scene: 8
  texture: 8

Metadata: /kaggle/working/pexels_metadata.json


In [10]:
import shutil
shutil.make_archive("/kaggle/working/real_videos_pexels", "zip", "/kaggle/working/pexels_normalised")
!ls -la /kaggle/working/*.zip /kaggle/working/*.json
print("\nReady to download real_videos_pexels.zip and pexels_metadata.json")
print("from the Output panel on the right sidebar.")

-rw-r--r-- 1 root root    49315 Jul 20 09:12 /kaggle/working/pexels_metadata.json
-rw-r--r-- 1 root root 19863889 Jul 20 09:12 /kaggle/working/real_videos_pexels.zip

Ready to download real_videos_pexels.zip and pexels_metadata.json
from the Output panel on the right sidebar.
